# Session 3 - The Central Limit Theorem, live from the room

> **Facilitator demo.** Run this after the Vevox 'countries visited' poll (Poll 2). It turns the room's own answers into the bell-shaped picture on the slide.

The one idea: **your individual answer is all over the place; the *average* of a sample is predictable - and gets tighter as the sample grows.** That is why we can trust a regression coefficient: it is an average-like thing with a bell around it.

**No poll data yet?** Leave `DATA_URL` blank and don't upload anything - the notebook falls back to a realistic stand-in (`room_data.csv`, from the course's public repo) so the demo always runs.

In [ ]:
# --- Load the room's answers --------------------------------------
# After the Vevox poll: Download responses -> Excel, save as room_data.csv, then either
#   Option A: paste a hosted link into DATA_URL, or
#   Option B: leave it blank and upload the file when prompted (Colab).
DATA_URL = ""

import pandas as pd, numpy as np

STAND_IN = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session3/data/room_data.csv"   # a realistic stand-in, if there is no export yet

if DATA_URL:
    raw = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload your Vevox export (or Cancel to use the stand-in) ...")
        up = files.upload()
        raw = pd.read_csv(next(iter(up))) if up else pd.read_csv(STAND_IN)
    except ImportError:
        raw = pd.read_csv('data/room_data.csv')   # running locally, not in Colab

# Vevox exports vary - grab the most-numeric column and keep sensible answers.
num = raw.apply(lambda c: pd.to_numeric(c, errors='coerce'))
col = num.notna().sum().idxmax()
room = num[col].dropna()
room = room[(room >= 1) & (room <= 80)].astype(int).values
print(f'{len(room)} answers loaded  |  mean {room.mean():.1f}  |  range {room.min()}-{room.max()}')

## 1. The room is lopsided, not a bell

This is the **population** for today: every answer in the room. Real data rarely looks like a bell.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.4))
plt.hist(room, bins=range(1, room.max() + 2), color='#606060', alpha=0.85)
plt.title('Countries visited - the whole room'); plt.xlabel('countries'); plt.ylabel('students')
plt.show()

## 2. Now average a sample - and watch the bell appear

We repeatedly draw a random sample of `n` people, take the **average**, and plot those averages. Same room, three sample sizes. Watch two things: the shape becomes a **bell**, and it gets **narrower** as `n` grows.

In [ ]:
def sample_means(pop, n, reps=2000, rng=np.random.default_rng(0)):
    return rng.choice(pop, size=(reps, n), replace=True).mean(axis=1)

sizes = [2, 10, 30]
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].hist(room, bins=range(1, room.max() + 2), color='#606060', alpha=0.85)
axes[0].set_title('The raw data')
for ax, n in zip(axes[1:], sizes):
    m = sample_means(room, n)
    ax.hist(m, bins=30, color='#1F497D', alpha=0.85)
    ax.set_title(f'Average of {n}')
    ax.set_xlim(0, room.mean() * 2.2)
for ax in axes:
    ax.set_yticks([])
plt.tight_layout(); plt.show()

## 3. The number behind the picture

The **average** of the averages barely moves off the room's true mean - but the **spread shrinks**. Roughly, four times the sample size halves the wobble (the square-root law).

In [ ]:
print(f'Room mean: {room.mean():.2f}\n')
print(f"{'sample size':>12} | {'mean of averages':>16} | {'spread (std)':>12}")
for n in [1, 2, 10, 30, 100]:
    m = sample_means(room, n)
    print(f'{n:>12} | {m.mean():>16.2f} | {m.std():>12.2f}')

## 4. The payoff (say this out loud)

- One person's answer is **wild**. The average of a sample is **boringly predictable** - and tighter the more data you have.
- That bell is exactly what sits around a **regression coefficient**: its standard error is the bell's width, and a confidence interval is that bell written as a plain-English range.
- **More (fair) data earns stronger claims** - a narrower bell. But a biased sample just gives you a confident *wrong* answer: precision is not the same as being right.

**Do it conversationally instead?** Paste the room's numbers into Claude:

> *"Here are the real answers from my class - how many countries each student has visited. Plot the histogram. Then repeatedly draw random samples of size 5, 10 and 30, average each, and plot the distribution of those averages next to the raw data. Narrate what happens to the shape and the width."*